In [0]:
%run /Repos/dung_nguyen_hoang@mfcgd.com/Utilities/Functions

In [0]:
import pyspark.sql.functions as F
import pyspark.sql.types as T
from pyspark.sql.window import Window
import pandas as pd

lst_mthend = '2025-07-31'
launch_dt = '2025-09-30'

In [0]:
# Load AVY customer list (only year 3++)
avy_cvg_df = spark.sql(f"""
SELECT DISTINCT
    cvg.pol_num
   ,po_lnk.cli_num                                      AS po_num
   ,ins_lnk.cli_num                                     AS ins_num
   --,po.cli_nm                                           AS cli_name
   ,case when pln.prod_cat='U' then
         case when pol.plan_code_base in ('UL004','UL005','UL035','UL036') then 'UL-Single-Premium'
              when pol.plan_code_base not in ('UL004','UL005','UL035','UL036') and pol.plan_code_base like 'UL%' then 'ULI'
              when pol.plan_code_base like 'RUV%' then 'UILP'
              else 'UL-Others'
         end
         when pln.prod_cat='T' then
         case when pol.plan_code_base in ('PN001','PA007','PA008') then 'Single-Year'
              when pol.plan_code_base in ('FDB01','BIC01','BIC02','BIC03','BIC04') then 'Digital'
              when pol.plan_code_base in ('BHC9I','CA360','CN360','CX360','MI007') then 'Activator-Micro'
              else 'Traditional'
         end
         else 'Others'
    end product_category
   ,cvg.plan_code
   ,code.product_name
   --,cvg.vers_num
   --,cvg.cvg_stat_cd
   ,cvg.cvg_typ
   --,cvg.cvg_reasn
   --,cvg.bnft_dur
   --,cvg.cvg_prem
   ,pol.pmt_mode
   ,cvg.cvg_prem*12/pol.pmt_mode                        AS inforce_APE
   --,pol.crcy_code
   --,cvg.face_amt

   -- payment that customer is due to pay
   ,inv.csh_cpn_bal                                     AS cashcoupon_value
   ,TO_DATE(cvg.cvg_eff_dt)                             AS pol_eff_dt
   --,cvg.cvg_iss_dt
   ,TO_DATE(pol.pol_iss_dt)                             AS pol_iss_dt
   ,DENSE_RANK() OVER (
        PARTITION BY
            po_lnk.cli_num
        ORDER BY
            pol.pol_iss_dt
           ,pol.pol_num
    )                                                   AS purchase_order_base
   ,ceiling(
       datediff(cast(concat(YEAR('{lst_mthend}'),
       '-',
       month(pol.pol_eff_dt),
       '-',
       day(pol.pol_eff_dt)) AS DATE), 
       pol.pol_eff_dt)/365.25)-1                        AS policy_tenure
   ,YEAR(cvg.xpry_dt)                                   AS maturity_year
   ,pol.wa_cd_1                                         AS wagt_cd
   ,wa.comp_prvd_num                                    AS wagt_comp_prvd_num
   ,pol.agt_code                                        AS agt_code
   ,sa.comp_prvd_num                                    AS sagt_comp_prvd_num
   ,CASE
        WHEN sa.trmn_dt IS NOT NULL
            AND sa.comp_prvd_num IN ('01','04','97','98') THEN 'Orphaned'
        WHEN sa.comp_prvd_num = '01'
            AND (
                pol.agt_code = pol.wa_cd_1
                OR pol.agt_code = pol.wa_cd_2
                )                                           THEN 'Original Agent'
        WHEN sa.comp_prvd_num = '01'                        THEN 'Reassigned Agent'
        WHEN sa.comp_prvd_num = '04'                        THEN 'Orphaned-Collector'
        WHEN sa.comp_prvd_num IN ('97', '98')               THEN 'Orphaned-Agent is SM'
        ELSE 'Unknown'
    END                                                 AS cus_agt_rltnshp
    ,CASE 
        WHEN YEAR('{lst_mthend}') - YEAR(pol.pol_eff_dt) >= 3 THEN 1
        ELSE 0  
    END AS FY_ANV_COUPON_IND
    ,SUBSTRING(CAST(
        CONCAT(
            year('{lst_mthend}'), '-', 
            LPAD(month(pol.pol_eff_dt), 2, '0'), '-', 
            LPAD(day(pol.pol_eff_dt), 2, '0')
        ) AS DATE
    ),1,7)                                               AS avy_month
    ,pol.image_date                                      AS image_date
FROM vn_published_casm_cas_snapshot_db.tcoverages cvg 
INNER JOIN vn_published_casm_cas_snapshot_db.tpolicys pol
    ON cvg.pol_num = pol.pol_num AND cvg.image_date = pol.image_date
-- base product category
INNER JOIN hive_metastore.vn_published_cas_db.tplans pln 
    ON pol.PLAN_CODE_BASE = pln.PLAN_CODE 
        AND pol.VERS_NUM_BASE = pln.VERS_NUM
-- owner
INNER JOIN vn_published_casm_cas_snapshot_db.tclient_policy_links po_lnk
    ON pol.pol_num = po_lnk.pol_num AND pol.image_date = po_lnk.image_date
        AND po_lnk.link_typ = 'O'
INNER JOIN vn_published_casm_cas_snapshot_db.tclient_details po
    ON po_lnk.cli_num = po.cli_num AND po_lnk.image_date = po.image_date
-- insured
INNER JOIN vn_published_casm_cas_snapshot_db.tclient_policy_links ins_lnk -- insured number
    ON pol.pol_num = ins_lnk.pol_num AND pol.image_date = ins_lnk.image_date
        AND ins_lnk.link_typ = 'I'
INNER JOIN vn_published_casm_cas_snapshot_db.tclient_details po_2
    ON ins_lnk.cli_num = po_2.cli_num AND ins_lnk.image_date = po_2.image_date
INNER JOIN vn_published_casm_ams_snapshot_db.tams_agents wa
    ON pol.wa_cd_1 = wa.agt_code AND pol.image_date = wa.image_date
INNER JOIN vn_published_casm_ams_snapshot_db.tams_agents sa
    ON pol.agt_code = sa.agt_code AND pol.image_date = sa.image_date
-- get cash coupon payment value (due to pay)
INNER JOIN vn_published_cas_db.tpol_invnty_ctl inv
    ON cvg.pol_num = inv.pol_num
inner join vn_curated_campaign_db.vn_plan_code_map code
    on cvg.plan_code=code.plan_code
WHERE   pol.image_date = '{lst_mthend}'
    AND cvg.cvg_typ = 'B'
    AND cvg.cvg_reasn = 'O'
    -- AND cvg.cvg_stat_cd IN ('1')
    -- agency sizing
    AND wa.comp_prvd_num in ('01','97','98')
    AND sa.comp_prvd_num in ('01','04','97','98')	
    AND po.sex_code in ('F', 'M') --Individual policies
    --AND substr(to_date(pol.pol_eff_dt),1,7) not in ('2014-01','2017-01','2019-01','2021-01') -- Exclude Anniversary falling in Jan'2024
    --AND to_date(cvg.xpry_dt) >= '2025-02-01'    -- excluding policies maturing before Feb'25
    --AND MONTH(pol.pol_eff_dt) IN (1)
    AND YEAR(cvg.xpry_dt) > YEAR('{lst_mthend}') --excluding maturity
    AND YEAR('{lst_mthend}') - YEAR(pol.pol_eff_dt) >= 3 --IN (3, 5, 7, 10) -- 3rd, 5th, 7th and 10th year
    AND pol.pol_stat_cd IN ('1','3','5') -- only premium paying or fully-paid policies
""")

# Keep only those with "policy_tenure" of year 3 and above, excluding single year policies
avy_cvg_df = avy_cvg_df.filter(
    (~F.col('product_category').isin('Activator-Micro','Digital')) &
    F.col('avy_month').isNotNull()
)

# Apply Window function to remove duplications
window_spec = Window.partitionBy("po_num").orderBy(avy_cvg_df["pol_eff_dt"].desc())

# Add a row number to each partition
avy_cvg_df = avy_cvg_df.withColumn("row_num", F.row_number().over(window_spec))

# Filter to keep only the latest pol_num for each po_num
avy_df = avy_cvg_df.filter(avy_cvg_df["row_num"] == 1).drop("row_num")

# check_dup(avy_df, "po_num")

In [0]:
ape_df = spark.sql(f"""
WITH ape_dtl AS (
SELECT  pol.image_date, cpl.cli_num AS po_num,
        CAST(SUM(pol.mode_prem*12/pol.pmt_mode) AS INT) AS total_ape
FROM    vn_published_casm_cas_snapshot_db.tpolicys pol
-- PO
INNER JOIN vn_published_casm_cas_snapshot_db.tclient_policy_links cpl
    ON pol.pol_num = cpl.pol_num AND pol.image_date = cpl.image_date
        AND cpl.link_typ = 'O' AND cpl.REC_STATUS = 'A'
WHERE   pol.image_date = '{lst_mthend}'
    AND pol.pol_stat_cd IN ('1','3','5')
GROUP BY
        pol.image_date, cpl.cli_num               
)
SELECT  image_date, po_num,
        total_ape,
        CASE
            WHEN NVL(total_ape,0) <  16000 THEN '<16m VND'
            WHEN NVL(total_ape,0) <= 40000 THEN '16m - 40m VND'
            WHEN NVL(total_ape,0) <= 65000 THEN '41m - 65m VND'
            ELSE '>65m VND'
        END AS ape_cat
FROM    ape_dtl
""")

In [0]:
# Derive latest repurchase propensity scores and deciles
onboard_path = f"/mnt/prod/Curated/VN/Master/VN_CURATED_CUSTOMER_ANALYTICS_DB/ONBOARDED_CUSTOMER_SCORE/image_date={lst_mthend}"
existing_path = f"/mnt/prod/Curated/VN/Master/VN_CURATED_CUSTOMER_ANALYTICS_DB/EXISTING_CUSTOMER_SCORE/image_date={lst_mthend}"

new_leads_score = spark.read.format("parquet").load(onboard_path).select("po_num", "decile").orderBy("po_num", "decile")
existing_leads_score = spark.read.format("parquet").load(existing_path).select("po_num", "decile").orderBy("po_num", "decile")
leads_score = new_leads_score.union(existing_leads_score
).dropDuplicates(["po_num"]
)

In [0]:
# Load latest Agency Structure from MIS
loc_path = f"/mnt/lab/vn/project/scratch/adhoc/loc_to_psm_mapping_202505.xlsx"
sheet_name = "Structure"
cell_pos = "A1"
loc_to_psm = spark.read.format(
    "com.crealytics.spark.excel"
).option(
    "dataAddress", f"{sheet_name}!{cell_pos}"
).option(
    "header", "true"
).option(
    "treatEmptyValuesAsNulls", "true"
).option(
    "inferSchema", "true"
).option(
    "addColorColumns", "False"
).load(
    loc_path
).select(
    "loc_cd", "psm_name_8", "psm_name_9"
)

# Derive Agent status and Group
mpro_title = spark.sql(f"""
SELECT  agt.AGT_CODE AS agt_code,
        CASE WHEN stat_cd = '01' THEN 'INFORCE'
             WHEN stat_cd = '99' THEN 'TERMINATED'
        END AS agt_status,
        CASE WHEN trmn_dt IS NOT NULL THEN '4.Terminated'
             WHEN trmn_dt IS NULL THEN
                 CASE
                     WHEN comp_prvd_num IN ('01','02','08','96') THEN '1.Inforce'
                     WHEN comp_prvd_num = '04' THEN '2.CSC'
                     WHEN comp_prvd_num IN ('97','98') THEN '3.SM'
                     ELSE '5.Not-Agency'
                 END
        END AS agt_rltnshp,
        CASE WHEN trmn_dt IS NOT NULL OR comp_prvd_num IN ('04','97','98') THEN 1 ELSE 0 
        END AS unassigned_ind,
        COALESCE(
            CASE WHEN agt.MPRO_TITLE IS NOT NULL THEN
                CASE WHEN MDRT_DESC IN ('TOT', 'COT') THEN 'MDRT'
                     ELSE MDRT_DESC
                END
                ELSE 'Normal'
            END,
            MPRO_TITLE
        ) AS mpro_title,
        agt.LOC_CODE AS loc_cd,
        agt.image_date
FROM    vn_published_casm_ams_snapshot_db.tams_agents agt
WHERE   1=1
    AND agt.image_date = '{lst_mthend}'
""")


par_df = spark.read.format("parquet").load(
    f"/mnt/lab/vn/project/cpm/datamarts/TPARDM_MTHEND/"
).select(
    "agt_cd", "last_9m_pol", "last_3m_pol", "last_mth_pol", "image_date"
).withColumnRenamed(
    "agt_cd", "agt_code"
)

mpro_title = mpro_title.join(
    par_df, 
    on=["agt_code", "image_date"],
    how="left"
).join(
    loc_to_psm,
    "loc_cd",
    "left"
)

agt_df = mpro_title.withColumn(
    "agt_actvness",
    F.when(F.col("last_mth_pol") > 0, "1m active")
    .when(F.col("last_3m_pol") > 0, "3m active")
    .when(F.col("last_9m_pol") > 0, "9m active")
    .otherwise("SA")
).withColumn(
    "agt_segment",
    F.when(F.col("agt_actvness") != "SA",
        F.when(F.col("mpro_title") != "Normal", F.col("mpro_title"))
        .otherwise(F.col("agt_actvness"))
    ).otherwise(
        F.when(F.col("agt_rltnshp").isin("2.CSC","3.SM","4.Terminated"), "UCM")
        .otherwise("SA")
    )
).drop(
    "last_3m_pol", "last_9m_pol", "last_mth_pol", "agt_status", "agt_rltnshp", "agt_actvness"
)

In [0]:
agt_df.groupBy(
    "image_date",
    "agt_segment"
).agg(
    F.count("agt_code").alias("agent_count")
).display()

In [0]:
merged_df = avy_df.join(
    agt_df,
    on=["agt_code","image_date"],
    how="left"
).join(
    leads_score,
    on="po_num",
    how="left"
).join(
    ape_df,
    on=["po_num","image_date"],
    how="left"
).fillna(
    {
    "decile": 0,
    "ape_cat": "<16m VND"
    }
)

# Re-segmentize agent categories
merged_df = merged_df.withColumn(
  "decile_cat",
  F.when(F.col("decile").isin(1,2), "Decile 1-2")
  .when(F.col("decile") == 3, "Decile 3")
  .when(F.col("decile").isin(4,5), "Decile 4-5")
  .when(F.col("decile").isin(6,7,8,9,10), "Decile 6-10")
  .otherwise("Unscored base")
).withColumn(
    "agt_main_cat",
    F.when(
        F.col("cus_agt_rltnshp") == "Original Agent",
        "Original"
    ).when(
        F.col("cus_agt_rltnshp") == "Reassigned Agent",
        "Reassigned"
    ).otherwise(
        "UCM"
    )
).withColumn(
    "agt_sub_cat",
    F.when(
        F.col("agt_segment") == "MDRT",
        "MDRT"
    ).when(
        F.col("agt_segment").isin("P", "G", "S"), 
        "ManPRO"
    ).when(
        F.col("agt_segment").isin("1m active", "3m active", "9m active"),
        "Active"
    ).when(
        F.col("agt_segment") == "SA", 
        "SA"
    ).otherwise("UCM")
).withColumn(
    "agt_cat",
    F.when(F.col("agt_main_cat") == "UCM", "UCM")
    .otherwise(F.concat_ws("-", F.col("agt_main_cat"), F.col("agt_sub_cat")))
)

# merged_df.cache()
# check_dup(merged_df, "po_num")

In [0]:
# merged_df.groupBy(
#     "agt_main_cat",
#     "agt_sub_cat",
#     "agt_cat"
# ).agg(
#     F.count("po_num").alias("customer_count")
# ).orderBy(
#     F.desc("customer_count")
# ).display()

###6_Define Segment (Value) by grouping

In [0]:
# Filter data for the 'result' dataframe
result = merged_df.select(
    "po_num", "decile", "decile_cat", "ape_cat", "agt_main_cat", "agt_sub_cat", "agt_cat"
).withColumn(
    "value_segment",
    F.when(
        (F.col("decile").isin([1, 2])) & 
        (F.col("ape_cat") == '>65m VND'),
        "High Value"
    ).when(
        (F.col("decile").isin([3, 4, 5])) & 
        (F.col("ape_cat").isin(['<16m VND', '16m - 40m VND', '41m - 65m VND'])) & 
        (F.col("agt_main_cat") == "Original"),
        "Medium Value"
    ).otherwise("Low Value")
)

# Cache the DataFrame if you are using it multiple times
result.cache()

# Define the segmentation criteria
# def segmentate_customers(row):
#     # High Value: High propensity (decile_cat 1-2) and high APE (ape_cat "High APE")
#     if row['decile_cat'] in [1, 2] and row['ape_cat'] == '>65m VND':
#         return 'High Value'
#     # Medium Value: Medium propensity (decile_cat 3-5), low to medium APE (ape_cat "Med APE" or "Low APE"), and served by an Original agent
#     elif row['decile_cat'] in [3, 4, 5] and row['ape_cat'] in ['<16m VND', '16 - 40m VND', '41m - 65m VND'] and row['agt_cat'].startswith('Original AG'):
#         return 'Medium Value'
#     # Low Value: All other customers
#     else:
#         return 'Low Value'

# # Apply the segmentation criteria to the dataframe
# result['value_segment'] = result.apply(segmentate_customers, axis=1)

# Summarize/group by 'value_segment'
summary_df = result.groupBy(
    "value_segment",
    "agt_main_cat",
    "agt_sub_cat",
    "agt_cat",
    "decile_cat",
    "ape_cat"
).agg(
    F.count("po_num").alias("customer_count")
)

result.unpersist()
# Print the summary dataframe
display(summary_df)


In [0]:
summary_pd = summary_df.toPandas()

summary_pd.to_csv(
    "/dbfs/mnt/lab/vn/project/scratch/adhoc/adhoc_avy_tnps_analysis/csv/avy_nurturing_segments.csv",
    index=False,
    header=True
)